In [65]:
# Example in Python
import numpy as np
import pandas as pd
from faker import Faker

fake = Faker()

# Example distributions
disability_types = ["mobility", "vision", "hearing", "cognitive", "psychiatric", "chronic health"]
disability_weights = [0.3, 0.15, 0.15, 0.2, 0.1, 0.1]  # Approximate real-world distribution

accommodation_needs = [
    "flexible schedule", 
    "physical workspace modifications",
    "noise reduction", 
    "remote work", 
    "assistive technology",
    "modified training materials", 
    "interpreter services"
]

work_preferences = ["fully remote", "hybrid", "in-office"]
work_pref_weights = [0.3, 0.4, 0.3]

In [66]:
def generate_employee_profiles(num_profiles=1000):
    profiles = []
    
    for i in range(num_profiles):
        profile = {
            "id": i,
            "disability_type": np.random.choice(disability_types, p=disability_weights),
            "experience_years": np.random.choice(range(0, 21)),
            "education_level": np.random.choice(["high school", "associate", "bachelor", "master", "phd"]),
        }
        
        # Add related accommodations based on disability type
        if profile["disability_type"] == "cognitive":
            profile["accommodations"] = np.random.choice(
                ["physical workspace modifications", "assistive technology", "modified training materials", "noise reduction", "flexible schedule"],
                size=np.random.randint(2, 6),
                replace=False
            ).tolist()

        if profile["disability_type"] == "psychiatric":
            profile["accommodations"] = np.random.choice(
                ["physical workspace modifications", "noise reduction", "flexible schedule"],
                size=np.random.randint(1, 3),
                replace=False
            ).tolist()

        if profile["disability_type"] == "chronic health":
            profile["accommodations"] = np.random.choice(
                ["physical workspace modifications", "noise reduction", "flexible schedule", "remote work"],
                size=np.random.randint(1, 4),
                replace=False
            ).tolist()

        if profile["disability_type"] == "hearing":
            profile["accommodations"] = np.random.choice(
                ["physical workspace modifications", "assistive technology", "modified training materials", "noise reduction", "interpreter services"],
                size=np.random.randint(1, 5),
                replace=False
            ).tolist()
        
        # Add similar logic for other disability types
        if profile["disability_type"] == "vision":
            profile["accommodations"] = np.random.choice(
                ["physical workspace modifications", "remote work", "flexible schedule"],
                size=np.random.randint(1, 3),
                replace=False
            ).tolist()        

        if profile["disability_type"] == "mobility":
            profile["accommodations"] = np.random.choice(
                ["physical workspace modifications", "remote work", "flexible schedule"],
                size=np.random.randint(1, 3),
                replace=False
            ).tolist()      

        profile["work_preference"] = np.random.choice(work_preferences, p=work_pref_weights)      
          
        profiles.append(profile)
    

    return pd.DataFrame(profiles)

In [67]:
def generate_employer_profiles(num_employers=200):
    employers = []
    
    for i in range(num_employers):
        employer = {
            "id": i,
            "company_size": np.random.choice(["small", "medium", "large"]),
            "industry": np.random.choice(["tech", "healthcare", "finance", "retail", "manufacturing"]),
            "location": fake.city(),
            "remote_policy": np.random.choice(["remote-friendly", "hybrid", "in-office"]),
        }
        
        # Available accommodations
        num_accommodations = np.random.randint(2, len(accommodation_needs))
        employer["available_accommodations"] = np.random.choice(
            accommodation_needs, 
            size=num_accommodations, 
            replace=False
        ).tolist()
        
        if employer["remote_policy"] != "in-office":
            if "remote work" not in employer['available_accommodations']:
                employer['available_accommodations'].append("remote work")
        
        employers.append(employer)
    
    return pd.DataFrame(employers)

In [48]:
def generate_matches(employees, employers):
    matches = []
    
    for _, employee in employees.iterrows():
        for _, employer in employers.iterrows():
            # Calculate match score based on your domain expertise
            score = 0
            
            # Accommodation match
            # Add a check before trying to convert to a set
            if isinstance(employee["accommodations"], (list, tuple, str)):
                employee_accommodations = set(employee["accommodations"])
            else:
                # Handle the case where accommodations is not iterable
                employee_accommodations = set()  # Empty set as fallback
            employer_accommodations = set(employer["available_accommodations"])
            accommodation_match = len(employee_accommodations.intersection(employer_accommodations)) / len(employee_accommodations) if employee_accommodations else 1
            score += accommodation_match * 50  # Weighted heavily
            
            # Work arrangement match
            if (employee["work_preference"] == "fully remote" and employer["remote_policy"] == "remote-friendly") or \
               (employee["work_preference"] == "hybrid" and employer["remote_policy"] in ["remote-friendly", "hybrid"]) or \
               (employee["work_preference"] == "in-office"):
                score += 30
            
            # Add more matching criteria here
            
            # Generate outcome (successful match or not)
            success_probability = score / 100
            outcome = np.random.choice([1, 0], p=[success_probability, 1-success_probability])
            
            matches.append({
                "employee_id": employee["id"],
                "employer_id": employer["id"],
                "match_score": success_probability,
                "successful_match": outcome
            })
    
    return pd.DataFrame(matches)


In [68]:
# Generate data
employees = generate_employee_profiles(1000)
employers = generate_employer_profiles(200)
matches = generate_matches(employees, employers)

# Save to CSV
employees.to_csv("synthetic_employees.csv", index=False)
employers.to_csv("synthetic_employers.csv", index=False)
matches.to_csv("synthetic_matches.csv", index=False)

In [69]:
print(matches['successful_match'].value_counts())

successful_match
1    104511
0     95489
Name: count, dtype: int64


In [70]:
print(employees.head())
print(employers.head())

   id disability_type  experience_years education_level  \
0   0        mobility                20          master   
1   1        mobility                11       associate   
2   2         hearing                 9       associate   
3   3     psychiatric                 6             phd   
4   4        mobility                20             phd   

                       accommodations work_preference  
0                       [remote work]          hybrid  
1  [physical workspace modifications]       in-office  
2       [modified training materials]       in-office  
3                   [noise reduction]    fully remote  
4                       [remote work]       in-office  
   id company_size       industry            location    remote_policy  \
0   0       medium  manufacturing      Rasmussenshire           hybrid   
1   1        small         retail         North Lance  remote-friendly   
2   2        large     healthcare          East Kelly        in-office   
3   3        

In [73]:
def prepare_training_data_with_one_hot(employees, employers, matches):
    training_data = []
    
    # Get all possible accommodation types
    all_accommodation_types = set()
    for _, emp in employees.iterrows():
        if isinstance(emp['accommodations'], (list, tuple)):
            all_accommodation_types.update(emp['accommodations'])
    
    for _, match in matches.iterrows():
        employee = employees[employees['id'] == match['employee_id']].iloc[0]
        employer = employers[employers['id'] == match['employer_id']].iloc[0]
        
        features = {
            # Basic features
            'disability_type': employee['disability_type'],
            'experience_years': employee['experience_years'],
            'work_preference': employee['work_preference'],
            'company_size': employer['company_size'],
            'industry': employer['industry'],
            'remote_policy': employer['remote_policy'],
        }
        
        # Add accommodation binary flags
        emp_accommodations = employee['accommodations'] if isinstance(employee['accommodations'], (list, tuple)) else []
        emp_accommodations = set(emp_accommodations)
        
        employer_accommodations = employer['available_accommodations'] if isinstance(employer['available_accommodations'], (list, tuple)) else []
        employer_accommodations = set(employer_accommodations)
        
        for acc_type in all_accommodation_types:
            features[f'emp_needs_{acc_type}'] = int(acc_type in emp_accommodations)
            features[f'employer_provides_{acc_type}'] = int(acc_type in employer_accommodations)
        
        # Target variable
        features['match_score'] = match['match_score']
        
        training_data.append(features)
     
        
    return pd.DataFrame(training_data)

In [74]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor  # for regression/ranking
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.ensemble import GradientBoostingRegressor

training_data = prepare_training_data_with_one_hot(employees, employers, matches)
print(training_data.columns)

# Identify categorical columns that need encoding
categorical_features = ['disability_type', 'work_preference', 'company_size', 'industry', 'remote_policy']
# You might have other categorical columns to add to this list

# Create a preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough',  # Keep other columns as is
    force_int_remainder_cols=False 
)

# Split your data
X = training_data.drop('match_score', axis=1)
y = training_data['match_score']

# Split into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create the full pipeline with preprocessing and model
# Limit model complexity
# model = Pipeline(steps=[
#     ('preprocessor', preprocessor),
#     ('regressor', RandomForestRegressor(max_depth=3, n_estimators=10))
# ])

model = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(n_estimators=100, learning_rate=0.1))
])

# Train the model on training data only
model.fit(X_train, y_train)

# Evaluate on the test set
predictions = model.predict(X_test)
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)
print(f"Mean Squared Error: {mse}")
print(f"R² Score: {r2}")

Index(['disability_type', 'experience_years', 'work_preference',
       'company_size', 'industry', 'remote_policy',
       'emp_needs_noise reduction', 'employer_provides_noise reduction',
       'emp_needs_modified training materials',
       'employer_provides_modified training materials',
       'emp_needs_remote work', 'employer_provides_remote work',
       'emp_needs_physical workspace modifications',
       'employer_provides_physical workspace modifications',
       'emp_needs_flexible schedule', 'employer_provides_flexible schedule',
       'emp_needs_interpreter services',
       'employer_provides_interpreter services',
       'emp_needs_assistive technology',
       'employer_provides_assistive technology', 'match_score'],
      dtype='object')
Mean Squared Error: 0.007638811713266554
R² Score: 0.8653722978325065


In [58]:
print(training_data.columns)

Index(['disability_type', 'experience_years', 'work_preference',
       'company_size', 'industry', 'remote_policy',
       'emp_needs_noise reduction', 'employer_provides_noise reduction',
       'emp_needs_modified training materials',
       'employer_provides_modified training materials',
       'emp_needs_remote work', 'employer_provides_remote work',
       'emp_needs_physical workspace modifications',
       'employer_provides_physical workspace modifications',
       'emp_needs_flexible schedule', 'employer_provides_flexible schedule',
       'emp_needs_interpreter services',
       'employer_provides_interpreter services',
       'emp_needs_assistive technology',
       'employer_provides_assistive technology', 'match_score'],
      dtype='object')


In [32]:
print("Features used:", X.columns.tolist())

Features used: ['disability_type', 'experience_years', 'work_preference', 'company_size', 'industry', 'remote_policy', 'emp_needs_physical workspace modifications', 'employer_provides_physical workspace modifications', 'emp_needs_flexible schedule', 'employer_provides_flexible schedule', 'emp_needs_remote work', 'employer_provides_remote work']


In [59]:
print(training_data.head())

  disability_type  experience_years work_preference company_size    industry  \
0        mobility                16       in-office       medium      retail   
1        mobility                16       in-office       medium      retail   
2        mobility                16       in-office        large      retail   
3        mobility                16       in-office       medium  healthcare   
4        mobility                16       in-office        small  healthcare   

     remote_policy  emp_needs_noise reduction  \
0        in-office                          0   
1           hybrid                          0   
2  remote-friendly                          0   
3        in-office                          0   
4           hybrid                          0   

   employer_provides_noise reduction  emp_needs_modified training materials  \
0                                  1                                      0   
1                                  1                              

In [57]:
import pickle

# Assuming your model is trained and stored in a variable called 'model'
# Save the model to a file
with open('disability_employment_matching_model.pkl', 'wb') as file:
    pickle.dump(model, file)

print("Model saved successfully!")

Model saved successfully!
